In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [2]:
df = pd.read_csv('C:/Users/Stefan/SBC/Hate_Speech_Recon/Data/Processed/train_with_features.csv')
print(f"Dataset loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few tweets (raw):")
for i in range(3):
    print(f"{i+1}. {df['tweet'].iloc[i]}")

Dataset loaded: (24783, 10)
Columns: ['count', 'hate_speech_count', 'offensive_language_count', 'neither_count', 'class', 'tweet', 'class_name', 'tweet_length', 'word_count', 'annotation_sum']

First few tweets (raw):
1. !!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out...
2. !!!!! RT @mleew17: boy dats cold...tyga dwn bad for cuffin dat hoe in the 1st place!!
3. !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby4life: You ever fuck a bitch and she start to cry? You be confused as shit


In [3]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def clean_tweet(text):
    """
    Comprehensive tweet cleaning function
    """
    if pd.isna(text):
        return ""
    #Convert to string
    text = str(text)
    #Convert to lowercase
    text = text.lower()
    #Remove url
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    #Remove username
    text = re.sub(r'@\w+', '', text)
    #Remove hashtag
    text = re.sub(r'#', '', text)
    text = re.sub(r'&\w+;', '', text)
    #Remove numbers
    text = re.sub(r'\d+', '', text)
    #Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    #Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_text(text):
    """
    Tokenize text into words
    """
    if not text:
        return []
    try:
        tokens = word_tokenize(text)
        return tokens
    except:
        return text.split()

def remove_stopwords(tokens):
    """
    Remove stopwords from token list
    """
    return [word for word in tokens if word not in stop_words and len(word) > 2]

def lemmatize_tokens(tokens):
    """
    Lemmatize tokens
    """
    return [lemmatizer.lemmatize(word) for word in tokens]

def preprocess_pipeline(text):
    """
    Complete preprocessing pipeline
    """
    #Clean
    text = clean_tweet(text)
    #Tokenize
    tokens = tokenize_text(text)
    #Remove stopwords
    tokens = remove_stopwords(tokens)
    #Lemmatize
    tokens = lemmatize_tokens(tokens)
    #Join back to string
    return ' '.join(tokens)
print("Cleaning functions defined")

Cleaning functions defined


In [4]:
print("="*70)
print("PREPROCESSING EXAMPLES")
print("="*70)
test_samples = df['tweet'].head(5)
for i, tweet in enumerate(test_samples, 1):
    print(f"\nExample {i}:")
    print(f"  Original:  {tweet}")
    print(f"  Cleaned:   {preprocess_pipeline(tweet)}")

PREPROCESSING EXAMPLES

Example 1:
  Original:  !!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out...
  Cleaned:   woman shouldnt complain cleaning house man always take trash

Example 2:
  Original:  !!!!! RT @mleew17: boy dats cold...tyga dwn bad for cuffin dat hoe in the 1st place!!
  Cleaned:   boy dat coldtyga dwn bad cuffin dat hoe place

Example 3:
  Original:  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby4life: You ever fuck a bitch and she start to cry? You be confused as shit
  Cleaned:   dawg ever fuck bitch start cry confused shit

Example 4:
  Original:  !!!!!!!!! RT @C_G_Anderson: @viva_based she look like a tranny
  Cleaned:   look like tranny

Example 5:
  Original:  !!!!!!!!!!!!! RT @ShenikaRoberts: The shit you hear about me might be true or it might be faker than the bitch who told it to ya &#57361;
  Cleaned:   shit hear might true might faker bitch told


In [5]:
print("\n" + "="*70)
print("PREPROCESSING ALL TWEETS")
print("="*70)

print("\nCleaning tweets...")
df['tweet_cleaned'] = df['tweet'].apply(preprocess_pipeline)

print(f"\nPreprocessing complete!")
print(f"\nTotal tweets processed: {len(df):,}")

empty_count = (df['tweet_cleaned'].str.strip() == '').sum()
print(f"Empty tweets after cleaning: {empty_count}")

print(f"\nCleaned text length statistics:")
df['cleaned_length'] = df['tweet_cleaned'].str.len()
print(df['cleaned_length'].describe())


PREPROCESSING ALL TWEETS

Cleaning tweets...

Preprocessing complete!

Total tweets processed: 24,783
Empty tweets after cleaning: 3

Cleaned text length statistics:
count    24783.000000
mean        42.423355
std         22.867660
min          0.000000
25%         24.000000
50%         39.000000
75%         59.000000
max        130.000000
Name: cleaned_length, dtype: float64


In [6]:
print("\n" + "="*70)
print("BEFORE AND AFTER COMPARISON")
print("="*70)
samples = df.sample(5)
for idx, row in samples.iterrows():
    print(f"\nTweet ID: {idx}")
    print(f"  Class: {row['class_name']}")
    print(f"  BEFORE: {row['tweet'][:80]}...")
    print(f"  AFTER:  {row['tweet_cleaned'][:80]}...")


BEFORE AND AFTER COMPARISON

Tweet ID: 9115
  Class: Offensive Language
  BEFORE: First game first play. DAT can fuck my bitch...
  AFTER:  first game first play dat fuck bitch...

Tweet ID: 15065
  Class: Offensive Language
  BEFORE: RT @DrummerKid0328: Them shits was dry as a bitch! Had to ask for water and shit...
  AFTER:  shit dry bitch ask water shit...

Tweet ID: 5702
  Class: Offensive Language
  BEFORE: @cb_marie as u should that's jus a clutch ass spot bitch we in there for the tur...
  AFTER:  thats jus clutch as spot bitch turn...

Tweet ID: 7228
  Class: Offensive Language
  BEFORE: @tyracountryman nah gurl this pussy private &#128581;...
  AFTER:  nah gurl pussy private...

Tweet ID: 539
  Class: Neither
  BEFORE: "Only a couple of Red Sox have gotten past first base." I'm dying laughing. Mayb...
  AFTER:  couple red sox gotten past first base dying laughing maybe yank need teach game...


## Removing empty tweets

In [7]:
original_count = len(df)
df = df[df['tweet_cleaned'].str.strip() != '']
filtered_count = len(df)

print(f"\n" + "="*70)
print("DATA FILTERING")
print("="*70)
print(f"Original tweets: {original_count:,}")
print(f"After removing empty: {filtered_count:,}")
print(f"Removed: {original_count - filtered_count:,}")

print(f"\nClass distribution after filtering:")
print(df['class_name'].value_counts())


DATA FILTERING
Original tweets: 24,783
After removing empty: 24,780
Removed: 3

Class distribution after filtering:
class_name
Offensive Language    19189
Neither                4161
Hate Speech            1430
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

print("\n" + "="*70)
print("TRAIN-TEST SPLIT")
print("="*70)
X = df['tweet_cleaned']
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)
print(f"\nTraining set size: {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Testing set size: {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)")

print(f"\nTraining set class distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTesting set class distribution:")
print(y_test.value_counts().sort_index())

print(f"\nClass proportions (Training vs Testing):")
for class_num in [0, 1, 2]:
    train_prop = (y_train == class_num).sum() / len(y_train) * 100
    test_prop = (y_test == class_num).sum() / len(y_test) * 100
    print(f"  Class {class_num}: Train {train_prop:.2f}% | Test {test_prop:.2f}%")



TRAIN-TEST SPLIT

Training set size: 19,824 (80.0%)
Testing set size: 4,956 (20.0%)

Training set class distribution:
class
0     1144
1    15351
2     3329
Name: count, dtype: int64

Testing set class distribution:
class
0     286
1    3838
2     832
Name: count, dtype: int64

Class proportions (Training vs Testing):
  Class 0: Train 5.77% | Test 5.77%
  Class 1: Train 77.44% | Test 77.44%
  Class 2: Train 16.79% | Test 16.79%


In [9]:
import os
train_df = pd.DataFrame({
    'tweet_cleaned': X_train,
    'class': y_train
})
test_df = pd.DataFrame({
    'tweet_cleaned': X_test,
    'class': y_test
})
train_df.to_csv('C:/Users/Stefan/SBC/Hate_Speech_Recon/Data/Processed/train_processed.csv', index=False)
test_df.to_csv('C:/Users/Stefan/SBC/Hate_Speech_Recon/Data/Processed/test_processed.csv', index=False)
print("\n" + "="*70)
print("DATA SAVED")
print("="*70)
print("Training data saved to: data/processed/train_processed.csv")
print("Testing data saved to: data/processed/test_processed.csv")


DATA SAVED
Training data saved to: data/processed/train_processed.csv
Testing data saved to: data/processed/test_processed.csv
